# Heart Disease Prediction

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('data/cleveland.csv')
df['disease'] = df['num'].apply(lambda x: True if x > 0 else False)

In [ ]:
df

#### Feature approximate ranking
- "age": 0.3691
- "trestbps": 0.4672
- "chol": 0.3773
- "thalach": 0.5176
- "oldpeak": 0.3790

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

features = ['thalach', 'trestbps', 'oldpeak', 'chol', 'age']

# Standardize features
for feature in features:
    df[feature] = (df[feature] - df[feature].mean()) / df[feature].std()

nn = KNeighborsClassifier()
params = {'n_neighbors': [1, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41]}
num_dimensions = range(1, 6)
for dim in num_dimensions:
    feat_df = df[features[:dim]]
    clf = GridSearchCV(nn, params, cv=10, scoring='f1')
    clf.fit(feat_df, df['disease'])
    print(f"Results for {dim} dimensions: Best params: {clf.best_params_}, Best score: {clf.best_score_}")

In [ ]:
features = ['thalach']

feat_df = df[features]

x_train, x_test, y_train, y_test = train_test_split(feat_df, df['disease'], test_size=0.2, random_state=63)

In [ ]:
from sklearn.model_selection import KFold
from sklearn.neighbors import NearestNeighbors

kfold = KFold(n_splits=10, shuffle=True, random_state=63)
precision_list = []
recall_list = []
f1_score_list = []
for i, (train_index, val_index) in enumerate(kfold.split(x_train, y_train)):
    print(f"Fold {i+1}:")
    x_df = x_train.iloc[train_index]
    x = x_df[features].values
    nn = NearestNeighbors(n_neighbors=37)
    fit = nn.fit(x)

    confusion_matrix = {
        "TP": 0,
        "TN": 0,
        "FP": 0,
        "FN": 0
    }

    for j in val_index:
        distances, indices = fit.kneighbors([x_train.iloc[j]])
        disease = 0
        healthy = 0
        for index in indices[0]:
            if y_train.iloc[index]:
                disease += 1
            else:
                healthy += 1
        if disease > healthy:
            if y_train.iloc[j]:
                confusion_matrix["TP"] += 1
            else:
                confusion_matrix["FP"] += 1
        else:
            if y_train.iloc[j]:
                confusion_matrix["FN"] += 1
            else:
                confusion_matrix["TN"] += 1

    print(f"Confusion Matrix: {confusion_matrix}")
    precision = confusion_matrix["TP"] / (confusion_matrix["TP"] + confusion_matrix["FP"]) if (confusion_matrix["TP"] + confusion_matrix["FP"]) > 0 else 0
    recall = confusion_matrix["TP"] / (confusion_matrix["TP"] + confusion_matrix["FN"]) if (confusion_matrix["TP"] + confusion_matrix["FN"]) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall)
    print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1_score:.4f}")

    precision_list.append(precision)
    recall_list.append(recall)
    f1_score_list.append(f1_score)

    print()

print(f"Average Precision: {sum(precision_list)/len(precision_list):.4f}")
print(f"Average Recall: {sum(recall_list)/len(recall_list):.4f}")
print(f"Average F1 Score: {sum(f1_score_list)/len(f1_score_list):.4f}")